In [1]:
# Download HITRAN24 data from web #

from pathlib import Path
import zipfile
from archnemesis.database.utils import fetch
from archnemesis.database.readers.hitran.isotopologues import download_hitran_isotope_data

# This is the directory where all files used by this notebook will be saved.
test_data_dir = Path("./test_data")

HITRAN_PF_URL_FMT = 'https://www.hitran.org/data/Q/q{global_id}.txt'

ans_pf_tabulated_database_file = test_data_dir / "hitran24_pf_tabulated.h5"
ans_pf_polynomial_database_file = test_data_dir / "hitran24_pf_poly.h5"

test_data_dir.mkdir(parents=True, exist_ok=True)

if not (test_data_dir / "isotope_data.py").exists():
	download_hitran_isotope_data(test_data_dir)

# A bit of fiddling to load the isotope data we just downloaded
import importlib
import sys

MODULE_PATH = test_data_dir / "isotope_data.py"
MODULE_NAME = "hitran_isotope_data"
spec = importlib.util.spec_from_file_location(MODULE_NAME, MODULE_PATH)
hitran_isotope_data = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = hitran_isotope_data 
spec.loader.exec_module(hitran_isotope_data)

hitran_isotope_dict = hitran_isotope_data.HitranIsotope.dict()



In [6]:
# Add HITRAN24 data to HDF5 file #

import numpy as np

from archnemesis.database.datatypes.hitran.gas_descriptor import HitranGasDescriptor
from archnemesis.database.datatypes.gas_descriptor import RadtranGasDescriptor

from archnemesis.database.data_holders.partition_function_data_holder import PartitionFunctionDataHolder
from archnemesis.database.datatypes.pf_data.tabulated_pf_data import TabulatedPFData
from archnemesis.database.datatypes.pf_data.polynomial_pf_data import PolynomialPFData

from archnemesis.database.filetypes.ans_partition_fn_data_file import AnsPartitionFunctionDataFile


USE_POLYNOMIAL_PARTITION_DATA = False

# Create variable to hold partition function data
pfdh_tabulated = PartitionFunctionDataHolder(
	'HITRAN24',
	'Data in this group is TABULATED and taken from the HITRAN24 database',
)

pfdh_polynomial = PartitionFunctionDataHolder(
	'HITRAN24',
	'Data in this group is POLYNOMIAL and taken from the HITRAN24 database',
)

# Loop over isotopologues and add their partition function data to the PartitionFunctionDataHolder instance
for (mol_id_ht, iso_id_ht), hitran_iso in hitran_isotope_dict.items():
	gas_desc = HitranGasDescriptor(mol_id_ht, iso_id_ht).to_radtran()

	if gas_desc is None:
		continue
	
	pf_data_str = fetch.file(HITRAN_PF_URL_FMT.format(global_id = hitran_iso.global_id))
	pf_array = np.fromstring(pf_data_str, dtype=float, sep=' ').reshape(-1,2)
	
	temp = pf_array[:,0]
	q = pf_array[:,1]
	
	
	tab_pf_data = TabulatedPFData(
		temp,
		q
	)
	
	pfdh_tabulated.add(
		gas_desc.gas_id,
		gas_desc.iso_id,
		tab_pf_data
	)
	
	pfdh_polynomial.add(
		gas_desc.gas_id,
		gas_desc.iso_id,
		PolynomialPFData(
			*tab_pf_data.as_poly()
		)
	)
	


# Get an HDF5 file to save the parttion function data in
hitran_pf_tabulated_data_file = AnsPartitionFunctionDataFile(ans_pf_tabulated_database_file)
hitran_pf_polynomial_data_file = AnsPartitionFunctionDataFile(ans_pf_polynomial_database_file)

# Add the data in `pdfh` to the HDF5 file, this makes virtual datasets in the `partition_function` top-level group that refer to the source
hitran_pf_tabulated_data_file.add_source_data(pfdh_tabulated.name, pfdh_tabulated, pfdh_tabulated.description)
hitran_pf_polynomial_data_file.add_source_data(pfdh_polynomial.name, pfdh_polynomial, pfdh_polynomial.description)

INFO :: file_in_chunks :: fetch.py-45 :: url='https://www.hitran.org/data/Q/q1.txt'
INFO :: file_in_chunks :: fetch.py-100 :: Fetching chunk 0. Chunk is 175.779296875 Kb. Fetched 0.0 Kb so far...
INFO :: file_in_chunks :: fetch.py-107 :: Fetch complete, downloaded 175.779296875 Kb in total over 1 chunks.
INFO :: file_in_chunks :: fetch.py-45 :: url='https://www.hitran.org/data/Q/q2.txt'
INFO :: file_in_chunks :: fetch.py-100 :: Fetching chunk 0. Chunk is 175.779296875 Kb. Fetched 0.0 Kb so far...
INFO :: file_in_chunks :: fetch.py-107 :: Fetch complete, downloaded 175.779296875 Kb in total over 1 chunks.
INFO :: file_in_chunks :: fetch.py-45 :: url='https://www.hitran.org/data/Q/q3.txt'
INFO :: file_in_chunks :: fetch.py-100 :: Fetching chunk 0. Chunk is 175.779296875 Kb. Fetched 0.0 Kb so far...
INFO :: file_in_chunks :: fetch.py-107 :: Fetch complete, downloaded 175.779296875 Kb in total over 1 chunks.
INFO :: file_in_chunks :: fetch.py-45 :: url='https://www.hitran.org/data/Q/q4.txt